# Transformer from Scratch
Source: https://www.datacamp.com/tutorial/building-a-transformer-with-py-torch

# Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import math
import copy

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: mps


# Basic Building Blocks:

![building-blocks-transformers](../images/building-blocks-transformers.png)

# Multi-head attention

In summary, the MultiHeadAttention class encapsulates the multi-head attention mechanism commonly used in transformer models. It takes care of splitting the input into multiple attention heads, applying attention to each head, and then combining the results. By doing so, the model can capture various relationships in the input data at different scales, improving the expressive ability of the model.

![Multi-head Attention Anatomy](../images/multi-head-attention.png)

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        # Ensure d_model is divisible by num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        # initialize dimensions
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Linear layers for tranforming inputs to queries, keys, and values
        self.W_q = nn.Linear(d_model, d_model)  # Query Transformation
        self.W_k = nn.Linear(d_model, d_model)  # Key Transformation
        self.W_v = nn.Linear(d_model, d_model)  # Value Transformation 
        self.W_o = nn.Linear(d_model, d_model)  # Output Transformation

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # Calculate attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # Apply mask if provided (used in decoder to prevent attending to future tokens during training)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        
        # Softmax to get attention weights/probabilities
        attn_probs = torch.softmax(attn_scores, dim=-1)

        # Multiply by values to get the final attention output
        attn_output = torch.matmul(attn_probs, V)
        return attn_output
    
    def split_heads(self, x):
        # Split the last dimension into (num_heads, d_k) and transpose to get shape (batch_size, num_heads, seq_len, d_k)
        # The split_heads method reshapes the input to have multiple heads for attention.
        batch_size, seq_length, _ = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
    
    def combine_heads(self, x):
        # Combine the heads back to (batch_size, seq_len, d_model)
        # The combine_heads method combines the multiple heads back to the original shape.
        batch_size, _, seq_length, _ = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))  # (batch_size, num_heads, seq_len, d_k)
        K = self.split_heads(self.W_k(K))  # (batch_size, num_heads, seq_len, d_k)
        V = self.split_heads(self.W_v(V))  # (batch_size, num_heads, seq_len, d_k)

        # Perform scaled dot-product attention
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Combine heads and apply output transformation
        output = self.W_o(self.combine_heads(attn_output))
        return output


# Position-wise feed-forward networks

the PositionWiseFeedForward class defines a position-wise feed-forward neural network that consists of two linear layers with a ReLU activation function in between. In the context of transformer models, this feed-forward network is applied to each position separately and identically. It helps in transforming the features learned by the attention mechanisms within the transformer, acting as an additional processing step for the attention outputs.

In [3]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        # self.fc1 and self.fc2: Two fully connected (linear) layers
        self.fc1 = nn.Linear(d_model, d_ff)  # First linear layer
        self.fc2 = nn.Linear(d_ff, d_model)  # Second linear layer
        # activation function, which introduces non-linearity between the two linear layers
        self.relu = nn.ReLU()  # ReLU activation function

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

# Positional Encoding

The PositionalEncoding class adds information about the position of tokens within the sequence. Since the transformer model lacks inherent knowledge of the order of tokens (due to its self-attention mechanism), this class helps the model to consider the position of tokens in the sequence.

In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()

        # pe: A tensor filled with zeros, which will be populated with positional encodings.
        pe = torch.zeros(max_seq_length, d_model)
        # position: A tensor containing the position indices for each position in the sequence
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        # div_term: A term used to scale the position indices in a specific way
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)  # Apply sine to even indices
        pe[:, 1::2] = torch.cos(position * div_term)  # Apply cosine to odd indices

        # pe is registered as a buffer, which means it will be part of the module's state but will not be considered a trainable parameter
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]    

# Building the Encoder Block

The EncoderLayer class defines a single layer of the transformer's encoder. It encapsulates a multi-head self-attention mechanism followed by the position-wise feed-forward neural network, with residual connections, layer normalization, and dropout applied as appropriate. Together, these components allow the encoder to capture complex relationships in the input data and transform them into a useful representation for downstream tasks. Typically, multiple such encoder layers are stacked to form the complete encoder part of a transformer model.

![Encoder Block](../images/encoder-block.png)

In [5]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Self-attention sublayer with residual connection and layer normalization
        attn_output = self.self_attn(x, x, x, mask)  # Self-attention
        x = self.norm1(x + self.dropout(attn_output))  # Add & Norm

        # Feed-forward sublayer with residual connection and layer normalization
        ff_output = self.feed_forward(x)  # Position-wise feed-forward
        x = self.norm2(x + self.dropout(ff_output))  # Add & Norm

        return x

# Decoder Block

Processing steps:
- Self-attention on target sequence: The input x is processed through a self-attention mechanism.
Add and normalize (after self-attention): The output from self-attention is added to the original x, followed by dropout and normalization using norm1.
- Cross-attention with encoder output: The normalized output from the previous step is processed through a cross-attention mechanism that attends to the encoder's output enc_output.
- Add and normalize (after cross-attention): The output from cross-attention is added to the input of this stage, followed by dropout and normalization using norm2.
- Feed-forward network: The output from the previous step is passed through the feed-forward network.
Add and normalize (after feed-forward): The feed-forward output is added to the input of this stage, followed by dropout and normalization using norm3.
- Output: The processed tensor is returned as the output of the decoder layer.


- ---
The DecoderLayer class defines a single layer of the transformer's decoder. It consists of a multi-head self-attention mechanism, a multi-head cross-attention mechanism (that attends to the encoder's output), a position-wise feed-forward neural network, and the corresponding residual connections, layer normalization, and dropout layers. This combination enables the decoder to generate meaningful outputs based on the encoder's representations, taking into account both the target sequence and the source sequence. As with the encoder, multiple decoder layers are typically stacked to form the complete decoder part of a transformer model.

![Decode Block](../images/decoder-block.png)

In [6]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, enc_output, src_mask, tgt_mask):
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))
        attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        return x

# Transformer

![Transformer Model](../images/transformer-model.png)

In [7]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout=0.1):
        super(Transformer, self).__init__()
        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

        self.fc = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, src, tgt):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3)
        seq_length = tgt.size(1)
        nopeak_mask = (1 - torch.triu(torch.ones(1, seq_length, seq_length, device=src.device), diagonal=1)).bool()
        tgt_mask = tgt_mask & nopeak_mask
        return src_mask, tgt_mask
    
    def forward(self, src, tgt):
        src_mask, tgt_mask = self.generate_mask(src, tgt)
        src_embedded = self.dropout(self.positional_encoding(self.encoder_embedding(src)))
        tgt_embedded = self.dropout(self.positional_encoding(self.decoder_embedding(tgt)))

        enc_output = src_embedded
        for enc_layer in self.encoder_layers:
            enc_output = enc_layer(enc_output, src_mask)

        dec_output = tgt_embedded
        for dec_layer in self.decoder_layers:
            dec_output = dec_layer(dec_output, enc_output, src_mask, tgt_mask)

        output = self.fc(dec_output)
        return output

# Preparing to Train the PyTorch Transformer Model

![Transformer Hyperparameters](../images/transformer-hyperparams.png)

In [8]:
src_vocab_size = 5000
tgt_vocab_size = 5000
d_model = 512
num_heads = 8
num_layers = 6
d_ff = 2048
max_seq_length = 100
dropout = 0.1

transformer = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout)
transformer = transformer.to(device)

# Generate random sample data
src_data = torch.randint(1, src_vocab_size, (64, max_seq_length)).to(device)  # (batch_size, seq_length)
tgt_data = torch.randint(1, tgt_vocab_size, (64, max_seq_length)).to(device)  # (batch_size, seq_length)

# Create a Transformer Instance

In [9]:
transformer = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout)
transformer = transformer.to(device)

# Training the Model

In [10]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(transformer.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

transformer.train()

for epoch in range(100):
    optimizer.zero_grad()
    output = transformer(src_data, tgt_data[:, :-1])
    loss = criterion(output.contiguous().view(-1, tgt_vocab_size), tgt_data[:, 1:].contiguous().view(-1))
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoch: {epoch+1}, Loss: {loss.item()}")

Epoch: 1, Loss: 8.684920310974121
Epoch: 11, Loss: 7.890628814697266
Epoch: 21, Loss: 7.062998294830322
Epoch: 31, Loss: 6.331636905670166
Epoch: 41, Loss: 5.67617654800415
Epoch: 51, Loss: 5.09115743637085
Epoch: 61, Loss: 4.559325695037842
Epoch: 71, Loss: 4.053534507751465
Epoch: 81, Loss: 3.578993082046509
Epoch: 91, Loss: 3.1274466514587402


# Evaluations

In [12]:
transformer.eval()

# Generate random sample validation data
val_src_data = torch.randint(1, src_vocab_size, (64, max_seq_length)).to(device)  # (batch_size, seq_length)
val_tgt_data = torch.randint(1, tgt_vocab_size, (64, max_seq_length)).to(device)  # (batch_size, seq_length)

with torch.no_grad():
    val_output = transformer(val_src_data, val_tgt_data[:, :-1])
    val_loss = criterion(val_output.contiguous().view(-1, tgt_vocab_size), val_tgt_data[:, 1:].contiguous().view(-1))
    print(f'Validation Loss: {val_loss.item()}')

Validation Loss: 8.831979751586914
